In [1]:
# importing basic libraries to handle data and files
import numpy as np
import pickle

# importing keras components to build and train the model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# for preparing text input for predictions
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# loading the processed datasets shared by Isha
X_train = np.load("X_train.npy")
X_val = np.load("X_val.npy")
X_test = np.load("X_test.npy")

y_train = np.load("y_train.npy")
y_val = np.load("y_val.npy")
y_test = np.load("y_test.npy")

# checking the shape of each dataset to confirm everything loaded correctly
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (1334, 200)
X_val: (286, 200)
X_test: (287, 200)
y_train: (1334,)
y_val: (286,)
y_test: (287,)


In [3]:
# loading the tokenizer that was already fitted during preprocessing
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# checking that it loaded correctly
print(type(tokenizer))

<class 'keras.src.legacy.preprocessing.text.Tokenizer'>


In [4]:
# getting the vocabulary size from the tokenizer
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)

Vocabulary size: 19251


In [5]:
# building the baseline LSTM model using the vocabulary size from Isha's tokenizer
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

In [6]:
# compiling the model for binary sentiment classification
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [7]:
# training the baseline model with the processed training and validation data
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)

Epoch 1/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 7s 118ms/step - accuracy: 0.4985 - loss: 0.6938 - val_accuracy: 0.5350 - val_loss: 0.6917
Epoch 2/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 145ms/step - accuracy: 0.5442 - loss: 0.6898 - val_accuracy: 0.5420 - val_loss: 0.6900
Epoch 3/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.5570 - loss: 0.6728 - val_accuracy: 0.5420 - val_loss: 0.6828
Epoch 4/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 124ms/step - accuracy: 0.6004 - loss: 0.6229 - val_accuracy: 0.5280 - val_loss: 0.7054
Epoch 5/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 127ms/step - accuracy: 0.6147 - loss: 0.5677 - val_accuracy: 0.4895 - val_loss: 0.7244


In [8]:
# evaluating the model on unseen test data to check final performance
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test accuracy:", test_accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5540 - loss: 0.6999
Test accuracy: 0.5540069937705994


In [12]:
# testing the trained model with a custom input sentence
sample_text = ["this product is amazing and I love it"]

# converting text to sequence using the same tokenizer
sample_seq = tokenizer.texts_to_sequences(sample_text)

# padding to match the input shape used during training
sample_pad = pad_sequences(sample_seq, maxlen=200, padding="post", truncating="post")

# getting prediction
prediction = model.predict(sample_pad)[0][0]

print("Prediction score:", prediction)

# interpreting the result
if prediction >= 0.5:
    print("Positive review")
else:
    print("Negative review")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Prediction score: 0.52329516
Positive review


In [13]:
# saving the trained model so it can be reused later in the web app
model.save("model.h5")